In [16]:
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain.agents import create_agent
from langchain.messages import HumanMessage
import os

# 大模型
llm = init_chat_model(
    "deepseek-chat", 
    temperature=0.1, 
    api_key=os.environ.get("DEEPSEEK_API"),
    base_url="https://api.deepseek.com/v1",
    model_provider="openai"
)

# 工具函数
@tool
def calulate(expression: str) -> str:
    """计算一个数学表达式的结果"""
    try:
        result = str(eval(expression))
    except Exception as e:
        result = f"计算错误: {e}"

@tool
def get_info(city_name: str) -> str:
    """获取城市信息

    Args:
        city_name (str): _description_

    Returns:
        str: _description_
    """
    city_data = {
        "北京": "北京是中国的首都，拥有丰富的历史和文化遗产。",
        "上海": "上海是中国最大的城市之一，以其现代化的城市景观和繁华的商业区而闻名。",
        "广州": "广州是中国南方的重要城市，以其美食和商业活动而著名。",
        "深圳": "深圳是中国改革开放的前沿城市，以其创新和高科技产业而闻名。",
        "杭州": "杭州是中国东部的一个美丽城市，以西湖和茶文化而闻名。"
    }
    return city_data.get(city_name, f"未找到关于{city_name}的信息。")

# 创建一个智能体
agent = create_agent(
    model=llm,
    tools=[calulate, get_info],
    system_prompt="你是一个智能助手，可以回答用户的问题，并使用工具函数来计算数学表达式或获取城市信息。"
)

from rich.panel import Panel
from rich.console import Console
console = Console()

# 测试智能体
user_input = "请计算 3 + 5 的结果"
response = agent.invoke({"messages": [HumanMessage(content=user_input)]})
console.print(Panel(f"[bold green]用户输入:[/bold green] {user_input}\n[bold yellow]最终回复:[/bold yellow] {response['messages'][-1].content}\n[bold orange]智能体回复:[/bold orange] {response}"))

console.print(Panel(title="消息列表", renderable="\n".join([f"[bold green]消息 {i}:[/bold green] {type(msg).__name__}: {msg}" for i, msg in enumerate(response['messages'])])))

# 查询城市信息

user_input = "请告诉我关于北京的信息"
response = agent.invoke({"messages": [HumanMessage(content=user_input)]})
console.print(Panel(f"[bold green]用户输入:[/bold green] {user_input}\n[bold yellow]最终回复:[/bold yellow] {response['messages'][-1].content}\n[bold orange]智能体回复:[/bold orange] {response}"))
console.print(Panel(title="消息列表", renderable="\n".join([f"[bold green]消息 {i}:[/bold green] {type(msg).__name__}: {msg}" for i, msg in enumerate(response['messages'])])))

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 用户输入: 请计算 3 + 5 的结果                                                                                   │
│ 最终回复: 3 + 5 的结果是 **8**。                                                                                │
│ 智能体回复: {'messages': [HumanMessage(content='请计算 3 + 5 的结果', additional_kwargs={},                     │
│ response_metadata={}, id='05ba8f93-b495-46f6-ac41-8d401fbc41fe'), AIMessage(content='好的，我来计算一下。',     │
│ additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 51,                │
│ 'prompt_tokens': 367, 'total_tokens': 418, 'completion_tokens_details': None, 'prompt_tokens_details':          │
│ {'audio_tokens': None, 'cached_tokens': 256}, 'prompt_cache_hit_tokens': 256, 'prompt_cache_miss_tokens': 111}, │
│ 'model_provider': 'openai', 'model_name': 'deepseek-v4-flash', 'system_fingerprint':                            │
│ 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': 'cdbbe7b7-9212-409d-ae93-94607af55d50', 'finish_reason':   │
│ 'tool_calls', 'logprobs': None}, id='lc_run--019f1e7e-fcca-7600-b76a-787db372b6ee-0', tool_calls=[{'name':      │
│ 'calulate', 'args': {'expression': '3 + 5'}, 'id': 'call_00_RJeeFkWqew1gCGy5syUD8534', 'type': 'tool_call'}],   │
│ invalid_tool_calls=[], usage_metadata={'input_tokens': 367, 'output_tokens': 51, 'total_tokens': 418,           │
│ 'input_token_details': {'cache_read': 256}, 'output_token_details': {}}), ToolMessage(content='null',           │
│ name='calulate', id='6c857da5-3b6e-4064-bf5b-edd1e8fd5cd2', tool_call_id='call_00_RJeeFkWqew1gCGy5syUD8534'),   │
│ AIMessage(content='3 + 5 的结果是 **8**。', additional_kwargs={'refusal': None},                                │
│ response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 431, 'total_tokens': 442,          │
│ 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 384},       │
│ 'prompt_cache_hit_tokens': 384, 'prompt_cache_miss_tokens': 47}, 'model_provider': 'openai', 'model_name':      │
│ 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id':                 │
│ 'b3104f76-a02f-464a-b3eb-2afd686a164c', 'finish_reason': 'stop', 'logprobs': None},                             │
│ id='lc_run--019f1e7f-018a-7803-81ce-5e248f7f946a-0', tool_calls=[], invalid_tool_calls=[],                      │
│ usage_metadata={'input_tokens': 431, 'output_tokens': 11, 'total_tokens': 442, 'input_token_details':           │
│ {'cache_read': 384}, 'output_token_details': {}})]}                                                             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────── 消息列表 ────────────────────────────────────────────────────╮
│ 消息 0: HumanMessage: content='请计算 3 + 5 的结果' additional_kwargs={} response_metadata={}                   │
│ id='05ba8f93-b495-46f6-ac41-8d401fbc41fe'                                                                       │
│ 消息 1: AIMessage: content='好的，我来计算一下。' additional_kwargs={'refusal': None}                           │
│ response_metadata={'token_usage': {'completion_tokens': 51, 'prompt_tokens': 367, 'total_tokens': 418,          │
│ 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256},       │
│ 'prompt_cache_hit_tokens': 256, 'prompt_cache_miss_tokens': 111}, 'model_provider': 'openai', 'model_name':     │
│ 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id':                 │
│ 'cdbbe7b7-9212-409d-ae93-94607af55d50', 'finish_reason': 'tool_calls', 'logprobs': None}                        │
│ id='lc_run--019f1e7e-fcca-7600-b76a-787db372b6ee-0' tool_calls=[{'name': 'calulate', 'args': {'expression': '3  │
│ + 5'}, 'id': 'call_00_RJeeFkWqew1gCGy5syUD8534', 'type': 'tool_call'}] invalid_tool_calls=[]                    │
│ usage_metadata={'input_tokens': 367, 'output_tokens': 51, 'total_tokens': 418, 'input_token_details':           │
│ {'cache_read': 256}, 'output_token_details': {}}                                                                │
│ 消息 2: ToolMessage: content='null' name='calulate' id='6c857da5-3b6e-4064-bf5b-edd1e8fd5cd2'                   │
│ tool_call_id='call_00_RJeeFkWqew1gCGy5syUD8534'                                                                 │
│ 消息 3: AIMessage: content='3 + 5 的结果是 **8**。' additional_kwargs={'refusal': None}                         │
│ response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 431, 'total_tokens': 442,          │
│ 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 384},       │
│ 'prompt_cache_hit_tokens': 384, 'prompt_cache_miss_tokens': 47}, 'model_provider': 'openai', 'model_name':      │
│ 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id':                 │
│ 'b3104f76-a02f-464a-b3eb-2afd686a164c', 'finish_reason': 'stop', 'logprobs': None}                              │
│ id='lc_run--019f1e7f-018a-7803-81ce-5e248f7f946a-0' tool_calls=[] invalid_tool_calls=[]                         │
│ usage_metadata={'input_tokens': 431, 'output_tokens': 11, 'total_tokens': 442, 'input_token_details':           │
│ {'cache_read': 384}, 'output_token_details': {}}                                                                │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 用户输入: 请告诉我关于北京的信息                                                                                │
│ 最终回复: 关于北京的信息如下：                                                                                  │
│                                                                                                                 │
│ **北京**是中国的首都，拥有丰富的历史和文化遗产。这里有许多著名的历史遗迹和景点，例如：                          │
│                                                                                                                 │
│ - 🏛️ **故宫博物院**（紫禁城）                                                                                   │
│ - 🏯 **天坛**                                                                                                   │
│ - 🚧 **长城**（八达岭、慕田峪等段）                                                                             │
│ - 🏞️ **颐和园**                                                                                                 │
│ - 🏟️ **天安门广场**                                                                                             │
│                                                                                                                 │
│ 此外，北京也是中国的政治、文化、教育和国际交往中心，是一座融合了古老传统与现代文明的国际化大都市。              │
│                                                                                                                 │
│ 如果您想了解更多关于北京的特定信息（比如人口、面积、美食等），欢迎继续提问！😊                                  │
│ 智能体回复: {'messages': [HumanMessage(content='请告诉我关于北京的信息', additional_kwargs={},                  │
│ response_metadata={}, id='9452c97b-c7a1-4379-84da-cac20b415654'),                                               │
│ AIMessage(content='好的，让我来查询北京的信息。', additional_kwargs={'refusal': None},                          │
│ response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 363, 'total_tokens': 415,          │
│ 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256},       │
│ 'prompt_cache_hit_tokens': 256, 'prompt_cache_miss_tokens': 107}, 'model_provider': 'openai', 'model_name':     │
│ 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id':                 │
│ '25ee4cf6-34f5-4792-9f14-9f13c44b8ffc', 'finish_reason': 'tool_calls', 'logprobs': None},                       │
│ id='lc_run--019f1e7f-05bb-7870-a042-39ddf4d19038-0', tool_calls=[{'name': 'get_info', 'args': {'city_name':     │
│ '北京'}, 'id': 'call_00_Ov5zgU9SwCnGUlO41ZQg5787', 'type': 'tool_call'}], invalid_tool_calls=[],                │
│ usage_metadata={'input_tokens': 363, 'output_tokens': 52, 'total_tokens': 415, 'input_token_details':           │
│ {'cache_read': 256}, 'output_token_details': {}}),                                                              │
│ ToolMessage(content='北京是中国的首都，拥有丰富的历史和文化遗产。', name='get_info',                            │
│ id='b2cc30bd-80e6-4346-a964-d1b3b63129ba', tool_call_id='call_00_Ov5zgU9SwCnGUlO41ZQg5787'),                    │
│ AIMessage(content='关于北京的信息如下：\n\n**北京**是中国的首都，拥有丰富的历史和文化遗产。这里有许多著名的历史 │
│ 遗迹和景点，例如：\n\n- 🏛️ **故宫博物院**（紫禁城）\n- 🏯 **天坛**\n- 🚧 **长城**（八达岭、慕田峪等段）\n- 🏞️   │
│ **颐和园**\n- 🏟️                                                                                                │
│ **天安门广场**\n\n此外，北京也是中国的政治、文化、教育和国际交往中心，是一座融合了古老传统与现代文明的国际化大  │
│ 都市。\n\n如果您想了解更多关于北京的特定信息（比如人口、面积、美食等），欢迎继续提问！😊',                      │
│ additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 135,               │
│ 'prompt_tokens': 436, 'total_tokens': 571, 'completion_tokens_details': None, 'prompt_tokens_details':          │
│ {'audio_tokens': None, 'cached_tokens': 256}, 'prompt_cache_hit_tokens': 256, 'prompt_cache_miss_tokens': 180}, │
│ 'model_provider': 'openai', 'model_name': 'deepseek-v4-flash', 'system_fingerprint':                            │
│ 'fp_8b330d0

╭─────────────────────────────────────────────────── 消息列表 ────────────────────────────────────────────────────╮
│ 消息 0: HumanMessage: content='请告诉我关于北京的信息' additional_kwargs={} response_metadata={}                │
│ id='9452c97b-c7a1-4379-84da-cac20b415654'                                                                       │
│ 消息 1: AIMessage: content='好的，让我来查询北京的信息。' additional_kwargs={'refusal': None}                   │
│ response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 363, 'total_tokens': 415,          │
│ 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256},       │
│ 'prompt_cache_hit_tokens': 256, 'prompt_cache_miss_tokens': 107}, 'model_provider': 'openai', 'model_name':     │
│ 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id':                 │
│ '25ee4cf6-34f5-4792-9f14-9f13c44b8ffc', 'finish_reason': 'tool_calls', 'logprobs': None}                        │
│ id='lc_run--019f1e7f-05bb-7870-a042-39ddf4d19038-0' tool_calls=[{'name': 'get_info', 'args': {'city_name':      │
│ '北京'}, 'id': 'call_00_Ov5zgU9SwCnGUlO41ZQg5787', 'type': 'tool_call'}] invalid_tool_calls=[]                  │
│ usage_metadata={'input_tokens': 363, 'output_tokens': 52, 'total_tokens': 415, 'input_token_details':           │
│ {'cache_read': 256}, 'output_token_details': {}}                                                                │
│ 消息 2: ToolMessage: content='北京是中国的首都，拥有丰富的历史和文化遗产。' name='get_info'                     │
│ id='b2cc30bd-80e6-4346-a964-d1b3b63129ba' tool_call_id='call_00_Ov5zgU9SwCnGUlO41ZQg5787'                       │
│ 消息 3: AIMessage:                                                                                              │
│ content='关于北京的信息如下：\n\n**北京**是中国的首都，拥有丰富的历史和文化遗产。这里有许多著名的历史遗迹和景点 │
│ ，例如：\n\n- 🏛️ **故宫博物院**（紫禁城）\n- 🏯 **天坛**\n- 🚧 **长城**（八达岭、慕田峪等段）\n- 🏞️             │
│ **颐和园**\n- 🏟️                                                                                                │
│ **天安门广场**\n\n此外，北京也是中国的政治、文化、教育和国际交往中心，是一座融合了古老传统与现代文明的国际化大  │
│ 都市。\n\n如果您想了解更多关于北京的特定信息（比如人口、面积、美食等），欢迎继续提问！😊'                       │
│ additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 135,                │
│ 'prompt_tokens': 436, 'total_tokens': 571, 'completion_tokens_details': None, 'prompt_tokens_details':          │
│ {'audio_tokens': None, 'cached_tokens': 256}, 'prompt_cache_hit_tokens': 256, 'prompt_cache_miss_tokens': 180}, │
│ 'model_provider': 'openai', 'model_name': 'deepseek-v4-flash', 'system_fingerprint':                            │
│ 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': 'e207a792-b084-46ff-8cf8-01349fc9a1ef', 'finish_reason':   │
│ 'stop', 'logprobs': None} id='lc_run--019f1e7f-0a87-7903-aaca-4d1339191a80-0' tool_calls=[]                     │
│ invalid_tool_calls=[] usage_metadata={'input_tokens': 436, 'output_tokens': 135, 'total_tokens': 571,           │
│ 'input_token_details': {'cache_read': 256}, 'output_token_details': {}}                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [16]:
# ReAct
from langchain.tools import tool
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain.chat_models import init_chat_model
import os

llm = init_chat_model(
    "deepseek-chat", 
    temperature=0.1,
    api_key=os.environ.get("DEEPSEEK_API"),
    base_url="https://api.deepseek.com/v1",
    model_provider="openai"
)

@tool
def calculate(expression: str) -> str:
    """计算一个数学表达式的结果
    When: 在需要计算数学表达式时使用此工具函数。
    How: 使用 Python 的 eval 函数计算表达式，并返回结果。输入必须满足 Python 的语法要求。
    
    Args:
        expression (str): 计算表达式

    Returns:
        str: 计算结果
    """
    try:
        result = str(eval(expression))
        return result
    except Exception  as e:
        return f"计算错误 : {e}"

@tool
def get_realtime_exchange_rate(currency: str) -> str:
    """获取实时汇率信息
    
    When: 当需要查询实时汇率时使用

    Args:
        currency (str): 货币代码，例如 USD、EUR、JPY 等

    Returns:
        str: 实时汇率信息
    """
    exchange_rates = {
        "USD": "1 USD = 6.5 CNY",
        "EUR": "1 EUR = 7.8 CNY",
        "JPY": "1 JPY = 0.06 CNY"
    }
    return exchange_rates.get(currency.upper(), f"未找到关于 {currency} 的汇率信息。")

agent = create_agent(
    model=llm,
    tools=[calculate, get_realtime_exchange_rate],
    system_prompt="你是一个专业金融计算助手，可以回答用户的问题，并使用工具函数来计算数学表达式或获取实时汇率信息。\n每一个回复前都要加上'尊敬的客户'"
)

question = "1500美元，可以兑换成多少人名币？"
answer = agent.invoke({"messages": [HumanMessage(content=question)]})

from pydantic import BaseModel
from rich.panel import Panel
from rich.console import Console, Group
console = Console()

console.print(Panel(answer["messages"][-1].content, title="Answer"))

group = Group(*[Panel(msg.model_dump_json(indent=2), title=f"{i}: {type(msg).__name__}") for i, msg in enumerate(answer["messages"])])
console.print(Panel(group, title="Messages"))

╭──────────────────────────────────────────────────── Answer ─────────────────────────────────────────────────────╮
│ 尊敬的客户，根据最新的实时汇率（1美元 = 6.5人民币），**1500美元可以兑换 9,750.00 人民币**。                     │
│                                                                                                                 │
│ 请注意，实际兑换时银行或兑换机构可能会收取一定的手续费，最终到账金额可能会略有差异。如果您需要查询其他货币的汇  │
│ 率，请随时告诉我！                                                                                              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────── Messages ────────────────────────────────────────────────────╮
│ ╭────────────────────────────────────────────── 0: HumanMessage ──────────────────────────────────────────────╮ │
│ │ {                                                                                                           │ │
│ │   "content": "1500美元，可以兑换成多少人名币？",                                                            │ │
│ │   "additional_kwargs": {},                                                                                  │ │
│ │   "response_metadata": {},                                                                                  │ │
│ │   "type": "human",                                                                                          │ │
│ │   "name": null,                                                                                             │ │
│ │   "id": "fab2b654-0a34-4953-9cc3-c6cbf14e47b3"                                                              │ │
│ │ }                                                                                                           │ │
│ ╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────╯ │
│ ╭─────────────────────────────────────────────── 1: AIMessage ────────────────────────────────────────────────╮ │
│ │ {                                                                                                           │ │
│ │   "content": "尊敬的客户，我来帮您查询实时汇率并进行计算。\n\n首先，让我获取美元兑人民币的实时汇率。",      │ │
│ │   "additional_kwargs": {                                                                                    │ │
│ │     "refusal": null                                                                                         │ │
│ │   },                                                                                                        │ │
│ │   "response_metadata": {                                                                                    │ │
│ │     "token_usage": {                                                                                        │ │
│ │       "completion_tokens": 70,                                                                              │ │
│ │       "prompt_tokens": 464,                                                                                 │ │
│ │       "total_tokens": 534,                                                                                  │ │
│ │       "completion_tokens_details": null,                                                                    │ │
│ │       "prompt_tokens_details": {                                                                            │ │
│ │         "audio_tokens": null,                                                                               │ │
│ │         "cached_tokens": 384                                                                                │ │
│ │       },                                                                                                    │ │
│ │       "prompt_cache_hit_tokens": 384,                                                                       │ │
│ │       "prompt_cache_miss_tokens": 80                                                                        │ │
│ │     },                                                                                                      │ │
│ │     "model_provider": "openai",                                                                             │ │
│ │     "model_name": "deepseek-v4-flash",                                                                      │ │
│ │     "system_fingerprint": "fp_8b330d02d0_prod0820_fp8_kvcache_20260402",                                    │ │
│ │     "id": "1cd140fb-01ba-4c44-a28e-e067dcb89ccf",                                                           │ │
│ │     "finish_reason": "tool_calls",                                                                         

In [19]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langchain.tools import tool
from pydantic import BaseModel, Field
from langchain.agents.structured_output import ToolStrategy
from langchain.chat_models import init_chat_model

llm = init_chat_model(
    "deepseek-chat", 
    temperature=0.1,
    api_key=os.environ.get("DEEPSEEK_API"),
    base_url="https://api.deepseek.com/v1",
    model_provider="openai"
)

class UserProfile(BaseModel):
    username: str = Field(description="用户名")
    aget: int = Field(description="用户年龄")
    is_active: bool = Field(description="用户是否有效")

@tool
def dummy_tool(query: str) -> str:
    """占位工具，保证有工具可以用
    """
    return "Tool is avaliable."

agent = create_agent(
    model=llm,
    tools=[dummy_tool],
    response_format=ToolStrategy(UserProfile),
    system_prompt="你是一位通用信息提取助理， 请使用 Tool Calling 机制提取信息"
)

result = agent.invoke({"messages": [HumanMessage(content="请你为我创建一个档案： 用户名是'Jane_D', 她今年22岁，账户当前是活跃阶段")]})
srtuct_result = result.get("structured_response")
print(srtuct_result)

from rich.panel import Panel
from rich.console import Console, Group
console = Console()

console.print(Panel(f"type: {type(srtuct_result).__name__}\ndata: {srtuct_result.model_dump_json(indent=2)}", title="Answer"))

group = Group(*[Panel(msg.model_dump_json(indent=2), title=f"{i}: {type(msg).__name__}") for i, msg in enumerate(result["messages"])])
console.print(Panel(group, title="Messages"))

username='Jane_D' aget=22 is_active=True


╭──────────────────────────────────────────────────── Answer ─────────────────────────────────────────────────────╮
│ type: UserProfile                                                                                               │
│ data: {                                                                                                         │
│   "username": "Jane_D",                                                                                         │
│   "aget": 22,                                                                                                   │
│   "is_active": true                                                                                             │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────── Messages ────────────────────────────────────────────────────╮
│ ╭────────────────────────────────────────────── 0: HumanMessage ──────────────────────────────────────────────╮ │
│ │ {                                                                                                           │ │
│ │   "content": "请你为我创建一个档案： 用户名是'Jane_D', 她今年22岁，账户当前是活跃阶段",                     │ │
│ │   "additional_kwargs": {},                                                                                  │ │
│ │   "response_metadata": {},                                                                                  │ │
│ │   "type": "human",                                                                                          │ │
│ │   "name": null,                                                                                             │ │
│ │   "id": "8e82adcf-ee69-4954-97f0-ae689b1dafb9"                                                              │ │
│ │ }                                                                                                           │ │
│ ╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────╯ │
│ ╭─────────────────────────────────────────────── 1: AIMessage ────────────────────────────────────────────────╮ │
│ │ {                                                                                                           │ │
│ │   "content": "",                                                                                            │ │
│ │   "additional_kwargs": {                                                                                    │ │
│ │     "refusal": null                                                                                         │ │
│ │   },                                                                                                        │ │
│ │   "response_metadata": {                                                                                    │ │
│ │     "token_usage": {                                                                                        │ │
│ │       "completion_tokens": 70,                                                                              │ │
│ │       "prompt_tokens": 410,                                                                                 │ │
│ │       "total_tokens": 480,                                                                                  │ │
│ │       "completion_tokens_details": null,                                                                    │ │
│ │       "prompt_tokens_details": {                                                                            │ │
│ │         "audio_tokens": null,                                                                               │ │
│ │         "cached_tokens": 384                                                                                │ │
│ │       },                                                                                                    │ │
│ │       "prompt_cache_hit_tokens": 384,                                                                       │ │
│ │       "prompt_cache_miss_tokens": 26                                                                        │ │
│ │     },                                                                                                      │ │
│ │     "model_provider": "openai",                                                                             │ │
│ │     "model_name": "deepseek-v4-flash",                                                                      │ │
│ │     "system_fingerprint": "fp_8b330d02d0_prod0820_fp8_kvcache_20260402",                                    │ │
│ │     "id": "65571655-a66b-4618-8f0c-886011fb95b6",                                                           │ │
│ │     "finish_reason": "tool_calls",                                               